In [1]:
import pandas as pd
import arviz as az
from matplotlib import pyplot as plt
import seaborn as sns
import pycountry
from matplotlib.pyplot import cm

from emu_renewal.inputs import OUTPUTS_PATH, get_country_mobility, get_standard_priors
from emu_renewal.outputs import load_targets, get_country_analyses, get_all_like_comps, get_param_vals_by_analysis

In [2]:
colours = cm.Dark2.colors

In [3]:
def plot_country_like_components(like_comps_dfs, proc_disp_samples, shared_disp_samples):
    comp_fig, axes = plt.subplots(4, 2, figsize=[11, 14])
    flat_axes = axes.ravel()
    for t, targ in enumerate(set(like_comps_dfs.columns.get_level_values(0))):
        ax = sns.kdeplot(like_comps_dfs[targ], fill=True, ax=flat_axes[t])
        ax.set_title(targ)
        flat_axes[t].get_legend().remove()
    sns.kdeplot(shared_disp_samples, fill=True, ax=flat_axes[-2])
    flat_axes[-2].set_title("shared_dispersion")    
    sns.kdeplot(proc_disp_samples, fill=True, ax=flat_axes[-1])
    flat_axes[-1].set_title("proc_dispersion")
    comp_fig.suptitle(country, fontsize=15)
    comp_fig.tight_layout()

In [4]:
job_path = OUTPUTS_PATH / "42780561"
countries = [p.parts[-1] for p in job_path.iterdir()]
like_comps = {}
for country in countries[:1]:
    country_path = job_path / country
    like_comps[country] = get_all_like_comps(country_path)

In [7]:
procs = {}
mobility = {}
proc_disp_samples = {}
fit_disp_samples = {}
spaghs = {}
targets = {}
for country in countries[:1]:

    country_path = job_path / country
    analyses = get_country_analyses(country_path)
    procs[country] = {a: pd.read_hdf(country_path / a / "spaghetti.h5")["process"] for a in analyses}
    
    mobility[country] = get_country_mobility(country)
    proc_disp_samples[country] = get_param_dist_df(country, job_path, "dispersion_proc")
    fit_disp_samples[country] = get_param_dist_df(country, job_path, "shared_dispersion")

    country_path = job_path / country
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    country_spaghs = [pd.read_hdf(country_path / a / "spaghetti.h5") for a in analyses]
    spaghs[country] = pd.concat(country_spaghs, keys=analyses, axis=1)
    targets[country] = load_targets(country_path / analyses[0])

NameError: name 'get_param_dist_df' is not defined